# Assignment 10

### Imports

In [28]:
import os
import torch
import torch.nn as nn
import numpy as np
import json
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append("../../MainProject/Assignment9")
from assignment9_functions import *

device = torch.device("cpu")
random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)

## Evaluate Mediapipe and our NN against kinect data

### Function for aligning mediapipe files with kinect files where frames have been excluded

In [32]:
def load_aligned_pair(mp_file, kinect_file, mp_dir, kinect_dir):
    df_mp = pd.read_csv(os.path.join(mp_dir, mp_file))
    df_kinect = pd.read_csv(os.path.join(kinect_dir, kinect_file))

    df_kinect.columns = df_kinect.columns.str.strip()

    # Align frames
    frames = df_kinect["FrameNo"].values
    df_mp = df_mp[df_mp["FrameNo"].isin(frames)]

    df_mp = df_mp.reset_index(drop=True)
    df_kinect = df_kinect.reset_index(drop=True)

    return df_mp, df_kinect

### Load kinect data and mediapipe data

In [38]:
# Load kinect data
kinect_data = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
train_files_kinect, test_files_kinect = split_csvfiles(kinect_data, random_seed, 0.9, 0)

# Load Mediapipe data
mp_data = "../../MainProject/Assignment10/data/csv_of_all_videos"
train_files_mp, test_files_mp = split_csvfiles(mp_data, random_seed, 0.9, 0)

# Align mediapipe and kinect training data
train_mp_list = []
train_kinect_list = []
for mp_file, kinect_file in zip(train_files_mp, train_files_kinect):
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)

    train_mp_list.append(df_mp)
    train_kinect_list.append(df_kinect)

train_data_mp = pd.concat(train_mp_list, ignore_index=True)
train_data_kinect = pd.concat(train_kinect_list, ignore_index=True)

# Align mediapipe and kinect testing data
test_mp_list = []
test_kinect_list = []
for mp_file, kinect_file in zip(test_files_mp, test_files_kinect):
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)

    test_mp_list.append(df_mp)
    test_kinect_list.append(df_kinect)

test_data_mp = pd.concat(test_mp_list, ignore_index=True)
test_data_kinect = pd.concat(test_kinect_list, ignore_index=True)



x_train_kinect, y_train_kinect = input_target_split(train_data_kinect)
x_test_kinect, y_test_kinect = input_target_split(test_data_kinect)


x_train_mp, y_train_mp = input_target_split(train_data_mp)
x_test_mp, y_test_mp = input_target_split(test_data_mp)

print("MP train rows:", len(x_train_mp))
print("Kinect train rows:", len(y_train_kinect))

MP train rows: 20269
Kinect train rows: 21520


### Train champion model on scaled mp data

In [37]:
X_train_mp = torch.tensor(x_train_mp.values, dtype=torch.float32).to(device)
Y_train_mp = torch.tensor(y_train_mp.values, dtype=torch.float32).to(device)
X_test_mp = torch.tensor(x_test_mp.values, dtype=torch.float32).to(device)
Y_test_mp = torch.tensor(y_test_mp.values, dtype=torch.float32).to(device)

X_train_kinect = torch.tensor(x_train_kinect.values, dtype=torch.float32).to(device)
Y_train_kinect = torch.tensor(y_train_kinect.values, dtype=torch.float32).to(device)
X_test_kinect = torch.tensor(x_test_kinect.values, dtype=torch.float32).to(device)
Y_test_kinect = torch.tensor(y_test_kinect.values, dtype=torch.float32).to(device)

# Load Champion model
model_root = "../z_pred_models"
metadata_path = os.path.join(model_root, "metadata", "champion_info.json")
model_path = os.path.join(model_root, "champion", "champion_model.pt")

# Load Champion model
with open(metadata_path, "r") as f:
    champion_info = json.load(f)

best_config = champion_info["hyperparameters"]

best_model = build_model(best_config, device)


# Train model on scaled mediapipe data to predict kinect data
loss_fn = nn.MSELoss()

result = train_one_model(
    best_model,
    best_config,
    X_train_mp,
    Y_train_kinect,
    X_train_mp,
    Y_train_kinect,
)
best_model.load_state_dict(result["best_state"])


# Evaluate NN on mediapipe (x, y) against corresponding kinetic z-data
test_loss, test_metrics, test_predictions = evaluate_model(
    best_model,
    X_test_mp,
    Y_test_kinect,
    loss_fn
)

mse_nn = test_metrics['mse']
mae_nn = test_metrics['mae']
r2_nn = test_metrics['r2']
bias_nn = test_metrics['bias']


# Evaluate Mediapipes z-pred against corresponding kinetic z-data
mse_mp = mean_squared_error(Y_test_mp, Y_test_kinect)
mae_mp = mean_absolute_error(Y_test_mp, Y_test_kinect)
r2_mp = r2_score(Y_test_mp, Y_test_kinect)
bias_mp = np.mean(Y_test_mp - Y_test_kinect)

metrics_table = pd.DataFrame({
    "Model": ["Neural Network", "MediaPipe"],
    "Loss(MSE)": [mse_nn, mse_mp],
    "MAE": [mae_nn, mae_mp],
    "Bias": [bias_nn, bias_mp],
    "R2": [r2_nn, r2_mp]
})

print(results_table.to_string(index=False))

c:\Users\Hugo Karsamo\Anaconda\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([21520, 13])) that is different to the input size (torch.Size([20269, 13])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (20269) must match the size of tensor b (21520) at non-singleton dimension 0